# Compiling Metadata for Project-k

In [6]:
%reload_ext autoreload
%autoreload 2

# Load dependecies
import pandas as pd
import os

study_level_metadata_file = "/Users/davidabelson/Library/CloudStorage/OneDrive-UniversityofCambridge/Code_repos/md_curation/data/raw/studies_combined_list.xlsx"

#Check the file exists - print that it does
if os.path.exists(study_level_metadata_file):
    print(f"File exists: {study_level_metadata_file}")
else:
    raise FileNotFoundError(f"Study level metadata file does not exist: {study_level_metadata_file}")

#Load the study level metadata
study_level_metadata = pd.read_excel(study_level_metadata_file)
display(study_level_metadata.head(3))




File exists: /Users/davidabelson/Library/CloudStorage/OneDrive-UniversityofCambridge/Code_repos/md_curation/data/raw/studies_combined_list.xlsx


,paper_title,paper_short_title,paper_link,Curator,kleb_assemblies_in_paper,isolates_in_study,study_accessions,sample_selection,ATB_location_prop,ATB_collection_date_prop,ATB_isolation_source_prop,ATB_host_prop
0,Epidemic of carbapenem-resistant Klebsiella pn...,EuSCAPE,https://pmc.ncbi.nlm.nih.gov/articles/PMC7244338/,David,2162,1718,PRJEB10018,AMR,0.770992,0.770992,0.769196,0.77
1,National genomic surveillance integrating stan...,enterobacterales_japan,https://pubmed.ncbi.nlm.nih.gov/38052776/,David,1240,1240,PRJDB10842,AMR,1.000000,0.999321,0.976239,1.00
2,Genome Sequencing Identifies Previously Unreco...,klebsiella_phillipines,https://academic.oup.com/cid/article/73/Supple...,David,259,862,PRJEB29738,AMR,0.000000,0.000000,0.000000,0.00


In [7]:
# Make new study_accessions column using the first accession in the list
study_level_metadata["study_accessions_clean"] = study_level_metadata["study_accessions"].str.split(",").str[0]
# Number of unique study accessions
print(f"Number of unique study accessions: {study_level_metadata['study_accessions_clean'].nunique()}")
# Display a list of duplicates
print("\nList of duplicates:")
print(study_level_metadata[study_level_metadata.duplicated(subset=['study_accessions_clean'])]["study_accessions_clean"])


Number of unique study accessions: 142

List of duplicates:
68      PRJEB27342
88     PRJNA288601
90     PRJNA475751
97      PRJEB10018
104     PRJEB27256
142     PRJEB29742
144     PRJEB29739
149    PRJNA548120
Name: study_accessions_clean, dtype: object


In [8]:
ENA_project_dir = "/Users/davidabelson/Library/CloudStorage/OneDrive-UniversityofCambridge/Aaron Weimann's files - project_k/data/raw/metadata/study_level_metadata/ENA_projects"

# Look for all subfolder names, report how many there are
subfolder_names = [f.name for f in os.scandir(ENA_project_dir) if f.is_dir()]
print(f"Number of subfolders: {len(subfolder_names)}")

# Report the subfolder names
print("\nSubfolder names:")
print(subfolder_names)

# Match these to the study_accessions_clean column
study_accessions_clean_matches = study_level_metadata["study_accessions_clean"].isin(subfolder_names)
print(f"Number of matches: {study_accessions_clean_matches.sum()}")

# Get unique study accessions from metadata
unique_study_accessions = set(study_level_metadata["study_accessions_clean"].unique())

# Find folders that don't have a match in metadata
# Need to check if folder name or any part of it matches
folders_without_match = []
for folder_name in subfolder_names:
    # Check if the folder name itself is in metadata
    if folder_name in unique_study_accessions:
        continue
    # Check if any accession extracted from folder name (split by comma, space, or underscore) is in metadata
    # Extract potential accessions (assuming they start with PRJ)
    folder_parts = folder_name.replace(',', ' ').replace('_', ' ').split()
    folder_accessions = [part for part in folder_parts if part.startswith('PRJ')]
    # Check if any of these accessions match
    if not any(acc in unique_study_accessions for acc in folder_accessions):
        folders_without_match.append(folder_name)

print(f"\nNumber of folders without match: {len(folders_without_match)}")
print("Folders without match:")
print(folders_without_match)

Number of subfolders: 95

Subfolder names:
['PRJEB6891', 'PRJEB37504', 'PRJNA646358', 'PRJEB39293', 'PRJDB12116', 'PRJNA565795', 'PRJEB39867', 'PRJNA376414', 'PRJEB6403_NCTC', 'PRJNA825705', 'PRJNA594602', 'PRJNA271899', 'PRJEB1963', 'PRJEB29740', 'PRJEB28400', 'PRJNA778230', 'PRJNA835677', 'PRJNA396774', 'PRJEB3255', 'PRJNA341927', 'PRJNA820335', 'PRJNA475751', 'PRJNA804332', 'PRJNA246471', 'PRJEB29143', 'PRJEB42462', 'PRJEB29739', 'PRJEB48990', 'PRJNA845975', 'PRJEB63349', 'PRJEB74192_ PRJNA1098507_complete_genomes', 'PRJNA548120', 'PRJEB27256', 'PRJNA658369', 'PRJEB10018', 'PRJEB29738', 'PRJNA339843', 'PRJEB38289', 'PRJNA996149', 'PRJEB21081', 'PRJNA396774_PRJNA376414_PRJNA386693', 'PRJNA767944', 'PRJNA288601', 'PRJEB58216', 'PRJEB29424', 'PRJNA557275', 'PRJEB50822', 'PRJNA543274', 'PRJNA514908', 'PRJNA564424', 'PRJNA1133668', 'PRJEB43870,PRJNA922900', 'PRJEB6891, PRJNA351909', 'PRJDB10842', 'PRJNA757551', 'PRJEB36486', 'PRJNA395086', 'PRJEB1271', 'PRJEB43870', 'PRJEB29740_PRJEB5061

In [9]:
# Of the folders with a match, check how many have files containing string "*ready_to_merge*" in them

# First, identify which folders have a match (using same logic as before)
matching_folders = []
for folder_name in subfolder_names:
    # Check if the folder name itself is in metadata
    if folder_name in unique_study_accessions:
        matching_folders.append(folder_name)
    else:
        # Check if any accession extracted from folder name matches
        folder_parts = folder_name.replace(',', ' ').replace('_', ' ').split()
        folder_accessions = [part for part in folder_parts if part.startswith('PRJ')]
        if any(acc in unique_study_accessions for acc in folder_accessions):
            matching_folders.append(folder_name)

print(f"Number of folders with match: {len(matching_folders)}")

# Now check each matching folder for files with "ready_to_merge" in the filename
from pathlib import Path

folders_with_ready_to_merge = []
folders_without_ready_to_merge = []

for folder_name in matching_folders:
    folder_path = os.path.join(ENA_project_dir, folder_name)
    
    # Check all files in the folder (recursively)
    has_ready_to_merge = False
    try:
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                if "ready_to_merge" in file:
                    has_ready_to_merge = True
                    break
            if has_ready_to_merge:
                break
    except Exception as e:
        print(f"Error checking folder {folder_name}: {e}")
        continue
    
    if has_ready_to_merge:
        folders_with_ready_to_merge.append(folder_name)
    else:
        folders_without_ready_to_merge.append(folder_name)

print(f"\nNumber of folders WITH 'ready_to_merge' files: {len(folders_with_ready_to_merge)}")
print(f"Number of folders WITHOUT 'ready_to_merge' files: {len(folders_without_ready_to_merge)}")

print("\nFolders WITHOUT 'ready_to_merge' files:")
print(list(folders_without_ready_to_merge))

Number of folders with match: 87

Number of folders WITH 'ready_to_merge' files: 63
Number of folders WITHOUT 'ready_to_merge' files: 24

Folders WITHOUT 'ready_to_merge' files:
['PRJNA646358', 'PRJNA594602', 'PRJNA778230', 'PRJNA835677', 'PRJNA341927', 'PRJNA475751', 'PRJNA246471', 'PRJEB29143', 'PRJNA658369', 'PRJNA396774_PRJNA376414_PRJNA386693', 'PRJNA288601', 'PRJNA557275', 'PRJEB50822', 'PRJEB43870', 'PRJEB29740_PRJEB50614', 'PRJEB24970', 'PRJEB42350', 'PRJEB33565', 'PRJNA415194, PRJNA252957', 'PRJNA789565', 'PRJEB2111', 'PRJEB24082', 'PRJNA634885', 'PRJEB56918']


In [10]:
# Check if any folders have more than one "ready_to_merge" file
for folder_name in folders_with_ready_to_merge:
    folder_path = os.path.join(ENA_project_dir, folder_name)
    ready_to_merge_files = [f for f in os.listdir(folder_path) if "ready_to_merge" in f]
    if len(ready_to_merge_files) > 1:
        print(f"Folder {folder_name} has more than one 'ready_to_merge' file:")


In [ ]:
# Lets load each of the files we find and merge them 